# 03/01 — Pseudobulk per (mouse, region)

Collapse spot-level counts to one row per (sample x region). Everything downstream (DGE, attenuation slope, mixed model) consumes this table, so the unit of replication is the *mouse* rather than the spot.

Outputs:
* `results/tables/attenuation/pseudobulk_counts.tsv`
* `results/tables/attenuation/pseudobulk_meta.tsv`

In [ ]:
from __future__ import annotations
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# regional-annotation h5ad lives on the processing volume
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'   # raw integer counts live here, not in .X

TBL = ROOT / 'results' / 'tables' / 'attenuation'
TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'
FIG.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY    = 'sample_id'           # change to 'library_id' if obs uses that
REGION_KEY    = 'anatomical_region'   # adjust to your obs column for regions
TREATMENT_KEY = 'treatment'
print('h5ad        :', H5AD)
print('count layer :', COUNT_LAYER)


In [ ]:
from utils.attenuation import make_pseudobulk

adata = sc.read_h5ad(H5AD)
print(adata)
for k in (SAMPLE_KEY, REGION_KEY, TREATMENT_KEY):
    if k not in adata.obs.columns:
        cands = [c for c in adata.obs.columns if k.split('_')[0] in c]
        print(f'!! {k!r} missing — candidates: {cands}')


### Sum spot counts to (sample, region)

Raw counts live in `adata.layers['counts']`. `make_pseudobulk` reads that layer directly — no need to mutate `.X`.

In [ ]:
counts, meta = make_pseudobulk(
    adata,
    sample_key=SAMPLE_KEY,
    region_key=REGION_KEY,
    treatment_key=TREATMENT_KEY,
    min_spots=10,            # was 20 — relax to keep small regions
    layer=COUNT_LAYER,
)
# cast to integer counts (matches DESeq2/edgeR expectations downstream)
counts = counts.round().astype(int)
print(counts.shape, meta.shape)
meta.head()


### Sanity

In [ ]:
print('mice per treatment:')
print(meta.groupby('treatment')['sample'].nunique())
print()
print('regions per treatment:')
print(meta.groupby(['treatment','region']).size().unstack(fill_value=0))


### Persist

In [ ]:
counts.to_csv(TBL / 'pseudobulk_counts.tsv', sep='\t')
meta.to_csv  (TBL / 'pseudobulk_meta.tsv',   sep='\t')
print('wrote', TBL / 'pseudobulk_counts.tsv')
print('wrote', TBL / 'pseudobulk_meta.tsv')
